In [5]:
%cd scripts

[Errno 2] No such file or directory: 'scripts'
/scratch/lgrinszt/lm_tab/scripts


In [3]:
%cd scripts

/scratch/lgrinszt/lm_tab/scripts


In [24]:
import sys
sys.path.append('/scratch/lgrinszt/carte')
from src_carte.carte_table_to_graph import Table2GraphTransformer
from src_carte.carte_estimator import CARTERegressor, CARTEClassifier
from configs.directory import config_directory


/scratch/lgrinszt/micromamba/envs/lm_tab/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning:

Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.



In [2]:
# find datasets where all ~ rest_only for all encoding methods
%cd scripts
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import numpy as np


melted_results = pd.concat([pd.read_csv("../results/results_limited_fixed_15_11.csv"), 
pd.read_csv("../results/results_skrub_varying_dims.csv"),
pd.read_csv("../results/results_new_datasets_29_08.csv"),
pd.read_csv("../results/results_new_datasets_29_08_2.csv"),
pd.read_csv("../results/complete_benchmark_results_02_09.csv")])
#melted_results = pd.read_csv("../results/results_all_n_train_5000.csv")

# restrict to n_train = 2000
#melted_results = melted_results[melted_results['n_train'] == 1000]

benchmark_datasets = ['bikewale', 'clear_corpus', 'company_employees',
       'employee-remuneration-and-expenses-earning-over-75000',
       'employee_salary', 'goodreads', 'journal_jcr_cls', 'ramen_ratings',
       'spotify', 'us_accidents_counts', 'us_accidents_severity',
       'us_presidential', 'wine_review', 'zomato']
autogluon_datasets = ['prod',
 'airbnb',
 'channel',
 'wine',
 'imdb',
 'jigsaw',
 'fake',
 'kick',
 'ae',
 'qaa',
 'qaq',
 'cloth',
 'mercari',
 'jc',
 'pop',
 'book',
 'salary',
 'house'
 ]
acceptable_datasets = benchmark_datasets + autogluon_datasets
melted_results = melted_results[melted_results["dataset"].isin(acceptable_datasets)]

melted_results = melted_results[melted_results["encoding"].isin(["openai__", "skrub__minhash_30", "lm__BAAI/bge-large-en-v1.5", "lm__intfloat/e5-large-v2",
"fasttext__30"])]

/scratch/lgrinszt/lm_tab/scripts


In [3]:
autogluon_results = pd.concat([pd.read_csv("../results/new_results_autogluon_29_08.csv"), 
                                pd.read_csv("../results/new_results_autogluon_29_08_2.csv"),
                                pd.read_csv("../results/new_results_autogluon_29_08_3.csv"),
                                pd.read_csv("../results/new_results_autogluon_29_08_not_binary.csv"),
                                pd.read_csv("../results/new_results_autogluon_30_08_10m.csv"),
                                pd.read_csv("../results/new_results_autogluon_31_08_best.csv")])

# remove missing accuracies
autogluon_results = autogluon_results[autogluon_results["accuracies"].notna()]
# I messed up a bit how I saved  this one, but I can get it
file_paths = ["../results/new_results_autogluon_e2.csv", "../results/missing_results_autogluon_e2_02_09.csv"]

# Initialize lists to store rows based on the number of columns
data_2_cols = []
data_3_cols = []
data_4_cols = []
data_7_cols = []

common_cols = ["n_train","n_test","features","dataset","time_limit","preset","hf_model","encoding"]
for file_path in file_paths:
    # Read the file line by line
    with open(file_path, 'r') as file:
        for line in file:
            # Split the line into columns
            columns = line.strip().split(',')
            num_columns = len(columns)

            # Check if the line is not the header
            if columns[0] == "accuracy":
                continue
            
            # Append the row to the appropriate list based on the number of columns
            if num_columns == 2 + len(common_cols):
                data_2_cols.append(columns)
            elif num_columns == 3 + len(common_cols):
                data_3_cols.append(columns)
            elif num_columns == 4 + len(common_cols):
                data_4_cols.append(columns)
            elif num_columns == 7 + len(common_cols):
                data_7_cols.append(columns)
            else:
                print(f"Unexpected number of columns: {num_columns}")

# Convert lists to DataFrames with appropriate column names
df_2_cols = pd.DataFrame(data_2_cols, columns=['accuracy', 'balanced_accuracy'] + common_cols)
df_3_cols = pd.DataFrame(data_3_cols, columns=['accuracy', 'balanced_accuracy', 'mcc'] + common_cols)
df_4_cols = pd.DataFrame(data_4_cols, columns=['accuracy', 'f1', 'roc_auc', 'balanced_accuracy'] + common_cols)
df_7_cols = pd.DataFrame(data_7_cols, columns=['accuracy', 'balanced_accuracy', 'mcc', 'roc_auc', 'f1', 'precision', 'recall'] + common_cols)
autogluon_results["encoding"] = "autogluon"# + autogluon_results["time_limit"].astype(str)
new_autogluon_results = pd.concat([df_2_cols, df_3_cols, df_4_cols, df_7_cols])
autogluon_results_full = pd.concat([autogluon_results, new_autogluon_results]).reset_index(drop=True)
# Replace missing hf_model with "default"
autogluon_results_full['hf_model'].fillna('default', inplace=True)
autogluon_results_full["encoding"] = autogluon_results_full["encoding"] + autogluon_results_full["time_limit"].astype(str) + autogluon_results_full["hf_model"]
# replace autogluon_180_lm__intfloat/e5-large-v2 with autogluon_180_lm__default because the parameter was not used
autogluon_results_full.loc[(autogluon_results_full['encoding'] == 'autogluon180intfloat/e5-large-v2'), 'encoding'] = 'autogluon180default'
#TODO: remove duplicates
melted_results = pd.concat([melted_results, autogluon_results_full])



/tmp/ipykernel_3222655/4166058262.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  autogluon_results_full['hf_model'].fillna('default', inplace=True)


In [4]:
carte_results = pd.concat([pd.read_csv("../results/results_carte_30_08_2.csv"),
                            pd.read_csv("../results/results_carte_30_08_3.csv"),
                            pd.read_csv("../results/results_carte_30_08_4.csv"),
                            pd.read_csv("../results/results_carte_01_09.csv")])
carte_results["encoding"] = "carte"
melted_results = pd.concat([melted_results, carte_results])


In [5]:
llm_results = pd.concat([pd.read_csv("../results/results_llm_30_08.csv"),
                         pd.read_csv("../results/results_llm_31_08.csv"),
                         pd.read_csv("../results/results_llm_30_08_text_only.csv")])
llm_results["encoding"] = "llm_icl"
melted_results = pd.concat([melted_results, llm_results])

#melted_results = melted_results[(melted_results["dim_reduction"].isna()) | (melted_results["dim_reduction"] == "passthrough")]
melted_results = melted_results[(melted_results["model"].isna()) | (melted_results["model"] == "GradientBoostingClassifier")]
#melted_results = melted_results[(melted_results["features"] == "text_only") | (melted_results["encoding"] == "llm_icl")]
melted_results = melted_results[(melted_results["features"] == "all")]

# Replace null accuracies with the values in the column "accuracies"
melted_results['accuracy'].fillna(melted_results['accuracies'], inplace=True)



/tmp/ipykernel_3222655/2178494642.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  melted_results['accuracy'].fillna(melted_results['accuracies'], inplace=True)


In [6]:
all_encodings

NameError: name 'all_encodings' is not defined

In [26]:
melted_results["encoding"].unique()

array(['skrub__minhash_30', 'openai__', 'lm__BAAI/bge-large-en-v1.5',
       'lm__intfloat/e5-large-v2', 'fasttext__30', 'autogluon180default',
       'autogluon_multimodal180intfloat/e5-large-v2',
       'autogluon_multimodal180default', 'carte', 'llm_icl'], dtype=object)

In [11]:
import plotly.express as px

group_cols = ['dataset', 'encoding', 'n_train', 'n_test', "dim_reduction", "features", "model", "preset",
"time_limit", "hf_model"]

melted_results[group_cols] = melted_results[group_cols].fillna("missing")

# Convert precision and recall columns to float
melted_results['precision'] = melted_results['precision'].astype(float)
melted_results['recall'] = melted_results['recall'].astype(float)
melted_results["roc_auc"] = melted_results["roc_auc"].astype(float)
melted_results["balanced_accuracy"] = melted_results["balanced_accuracy"].astype(float)
melted_results["f1"] = melted_results["f1"].astype(float)
melted_results["mcc"] = melted_results["mcc"].astype(float)
melted_results["accuracy"] = melted_results["accuracy"].astype(float)
melted_results["n_train"] = melted_results["n_train"].astype(int)
melted_results["n_test"] = melted_results["n_test"].astype(int)


to_plot = melted_results[melted_results["n_train"] == 5000]
to_plot = to_plot[to_plot['dataset'].isin(autogluon_datasets)]


all_encodings = list(to_plot.encoding.unique())
# Remove 'llm_icl' and 'autogluon1800' from all_encodings
# all_encodings = [encoding for encoding in all_encodings if encoding not in ['llm_icl', 'autogluon1800', "autogluon600.0", 
#  'autogluon600.0default', 'autogluonnandefault', "lm__intfloat/e5-large-v2",
# 'autogluon1800default', 'autogluonmedium_qualitydefault']]
all_encodings = ['skrub__minhash_30',
 'openai__',
 'lm__BAAI/bge-large-en-v1.5',
 'autogluon180default',
 "lm__intfloat/e5-large-v2",
 #'autogluon180intfloat/e5-large-v2',
 'autogluon_multimodal180default',
 'autogluon_multimodal180intfloat/e5-large-v2'
]
 #"carte",
 #"fasttext__30"]


# Select datasets where all encodings are present and each encoding has 7 rows
valid_datasets = []
for dataset in to_plot['dataset'].unique():
    print(dataset)
    dataset_subset = to_plot[to_plot['dataset'] == dataset]
    if all(np.isin(dataset_subset['encoding'].value_counts(), [7, 14, 21, 28])) and all(encoding in dataset_subset['encoding'].unique() for encoding in all_encodings):
        valid_datasets.append(dataset)
    else:
        print(f"Dataset {dataset} is not valid because it does not have all encodings or each encoding does not have exactly 7 rows.")
        print(f"Value counts for encodings in dataset {dataset}:")
        print(dataset_subset['encoding'].value_counts())
# Identify datasets that are missing any of the required encodings
missing_encodings = {}
for dataset in to_plot['dataset'].unique():
    dataset_subset = to_plot[to_plot['dataset'] == dataset]
    missing = [encoding for encoding in all_encodings if encoding not in dataset_subset['encoding'].unique()]
    if missing:
        missing_encodings[dataset] = missing

# Print the datasets with their missing encodings
for dataset, missing in missing_encodings.items():
    print(f"Dataset {dataset} is missing the following encodings: {missing}")
        
        

filtered_results = to_plot[to_plot['dataset'].isin(valid_datasets)]
filtered_results = filtered_results[filtered_results['encoding'].isin(all_encodings)]
#filtered_results = filtered_results[filtered_results['dataset'].isin(benchmark_datasets)]






grouped_results = filtered_results.groupby(group_cols).mean().reset_index()


px.strip(grouped_results, x="dataset", y="accuracy", color="encoding", facet_row="n_train",
         height=1000, width=1200)

channel
wine
jigsaw
fake
kick
ae
cloth
mercari
pop
salary


In [10]:
np.mean([0.89, 0.946, 0.938, 0.928, 0.886, 0.92, 0.956])

0.9234285714285715

In [178]:
melted_results[melted_results["dataset"] == "imdb"]

,encoding,dim_reduction,model,n_train,n_test,dataset,features,accuracies,roc_auc,time_limit,preset,accuracy,balanced_accuracy,mcc,f1,precision,recall,hf_model
2982,skrub__minhash_30,passthrough,GradientBoostingClassifier,64,500,imdb,all,0.630,0.698579,missing,missing,0.630,NaN,NaN,NaN,NaN,NaN,missing
2983,skrub__minhash_30,passthrough,GradientBoostingClassifier,64,500,imdb,all,0.582,0.665855,missing,missing,0.582,NaN,NaN,NaN,NaN,NaN,missing
2984,skrub__minhash_30,passthrough,GradientBoostingClassifier,64,500,imdb,all,0.630,0.701301,missing,missing,0.630,NaN,NaN,NaN,NaN,NaN,missing
2985,skrub__minhash_30,passthrough,GradientBoostingClassifier,64,500,imdb,all,0.558,0.598518,missing,missing,0.558,NaN,NaN,NaN,NaN,NaN,missing
2986,skrub__minhash_30,passthrough,GradientBoostingClassifier,64,500,imdb,all,0.628,0.703749,missing,missing,0.628,NaN,NaN,NaN,NaN,NaN,missing
2987,skrub__minhash_30,passthrough,GradientBoostingClassifier,64,500,imdb,all,0.530,0.543204,missing,missing,0.530,NaN,NaN,NaN,NaN,NaN,missing
2988,skrub__minhash_30,passthrough,GradientBoostingClassifier,64,500,imdb,all,0.510,0.554816,missing,missing,0.510,NaN,NaN,NaN,NaN,NaN,missing
3003,skrub__minhash_30,passthrough,GradientBoostingClassifier,128,500,imdb,all,0.626,0.720744,missing,missing,0.626,NaN,NaN,NaN,NaN,NaN,missing
3004,skrub__minhash_30,passthrough,GradientBoostingClassifier,128,500,imdb,all,0.662,0.739572,missing,missing,0.662,NaN,NaN,NaN,NaN,NaN,missing
3005,skrub__minhash_30,passthrough,GradientBoostingClassifier,128,500,imdb,all,0.628,0.697223,missing,missing,0.628,NaN,NaN,NaN,NaN,NaN,missing


In [176]:
len(autogluon_datasets) - len(valid_datasets)
# Find datasets that are in autogluon_datasets but not in valid_datasets
missing_datasets = set(autogluon_datasets) - set(valid_datasets)
print(f"Missing datasets: {missing_datasets}")


Missing datasets: {'prod', 'imdb', 'ae', 'mercari'}


In [12]:
# Calculate the mean rank per encoding
ranked_results = grouped_results.copy()
ranked_results['rank'] = ranked_results.groupby('dataset')['roc_auc'].rank(ascending=False)
mean_rank_per_encoding = ranked_results.groupby('encoding')['rank'].mean().reset_index()
mean_rank_per_encoding = mean_rank_per_encoding.sort_values(by='rank')
mean_rank_per_encoding


,encoding,rank
4,lm__intfloat/e5-large-v2,2.777778
3,lm__BAAI/bge-large-en-v1.5,2.888889
5,openai__,3.000000
0,autogluon180default,3.600000
1,autogluon_multimodal180default,4.800000
2,autogluon_multimodal180intfloat/e5-large-v2,5.800000
6,skrub__minhash_30,5.888889


In [6]:
unique_datasets_per_encoding = melted_results.groupby('encoding')['dataset'].nunique().reset_index()
unique_datasets_per_encoding


,encoding,dataset
0,autogluon180,31
1,autogluon1800,13
2,autogluon600.0,13
3,carte,19
4,llm_icl,10
5,lm__BAAI/bge-large-en-v1.5,31
6,lm__intfloat/e5-large-v2,17
7,openai__,31
8,skrub__minhash_30,32


In [19]:
X[["model_year"]].

model_year    int64
dtype: object

In [9]:
X

,Year,Name,Department,Title,Expenses
16704,2016,"Butt, B",Community Services,Planner I,60.00
23138,2013,"Devlin, J",Engineering Services,Equipment Operator V,0.00
7852,2019,"Darnell, B",Engineering Services,Journeyman - MacHinist,0.00
27464,2011,"Brown, K L",Engineering Services,Superintendent Il,0.00
13519,2017,"Lefkowitz, B",VFRS & OEM,Firefighter,0.00
...,...,...,...,...,...
28219,2011,"Harris, T A",Board of Parks & Recreation,Superintendent Park Board,0.00
8280,2019,"Mocharski, M R",Engineering Services,Working Foreman,0.00
23013,2013,"Antoniali, S M",Real Estate & Facilities Mgmt,Manager Property Management,1570.71
20236,2015,"Felder, R T",Engineering Services,Sewer Separation Expediter,0.00


In [10]:
from sklearn.preprocessing import PowerTransformer
#X_num = X[["model_year", "km_driven"]]
X_num = X[['Year', 'Expenses']]
# center
#X_num = X_num - X_num.mean()


X_num = PowerTransformer().fit_transform(X_num)



/scratch/lgrinszt/micromamba/envs/lm_tab/lib/python3.10/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/scratch/lgrinszt/micromamba/envs/lm_tab/lib/python3.10/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)


In [ ]:
X[["model_year"]]

NameError: name 'X' is not defined

In [23]:
X_num

array([[ 1.20368175],
       [-0.78006258],
       [-0.59286141],
       ...,
       [-0.78006258],
       [ 0.73262128],
       [-0.59286141]])

In [2]:
    %cd scripts
    import sys
    sys.path.append('/scratch/lgrinszt/carte')
    from src_carte.carte_table_to_graph import Table2GraphTransformer
    from src_carte.carte_estimator import CARTERegressor, CARTEClassifier
    from configs.directory import config_directory
    from src.data_loading import load_data
    from src.utils import FixedSizeSplit
    import pandas as pd
    import numpy as np
    from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score


    X, y = load_data("spotify", max_rows=10000)

    cv = FixedSizeSplit(n_splits=7, n_train=300, n_test=500, random_state=42)

    # Define some parameters
    fixed_params = dict()
    fixed_params["num_model"] = 2 # 10 models for the bagging strategy
    fixed_params["disable_pbar"] = False # True if you want cleanness
    fixed_params["random_state"] = 0
    fixed_params["device"] = "cpu"
    fixed_params["n_jobs"] = 10
    fixed_params["max_epoch"] = 2



    accs = []
    roc_aucs = []
    balanced_accs = []
    for train_idx, test_idx in cv.split(X):
        preprocessor = Table2GraphTransformer()
        X_train = X.iloc[train_idx]
        y_train = y[train_idx]
        X_test = X.iloc[test_idx]
        y_test = y[test_idx]
        X_train = preprocessor.fit_transform(X_train, y=y_train)
        X_test = preprocessor.transform(X_test)

        is_binary = len(np.unique(y)) == 2

        # Define the estimator and run fit/predict
        estimator = CARTEClassifier(**fixed_params,
        loss="binary_crossentropy" if is_binary else "categorical_crossentropy") # CARTERegressor for Regression
        estimator.fit(X=X_train, y=y_train)
        y_pred_proba = estimator.predict_proba(X_test)
        if is_binary:
            y_pred = y_pred_proba > 0.5
        else:
            y_pred = y_pred_proba.argmax(axis=1)

        # Obtain the r2 score on predictions
        try:
            score = roc_auc_score(y_test, y_pred_proba)
            print(f"\nThe AUROC for CARTE:", "{:.4f}".format(score))
            roc_aucs.append(score)
        except Exception as e:
            print(f"Error computing AUROC: {e}")
            roc_aucs.append(None)

        acc = accuracy_score(y_test, y_pred)
        print(f"\nThe accuracy for CARTE:", "{:.4f}".format(acc))
        accs.append(acc)

        balanced_acc = balanced_accuracy_score(y_test, y_pred)
        print(f"\nThe balanced accuracy for CARTE:", "{:.4f}".format(balanced_acc))
        balanced_accs.append(balanced_acc)

[Errno 2] No such file or directory: 'scripts'
/scratch/lgrinszt/lm_tab/scripts
Removed 0 columns with missing values on 17 columns
Removed 0 rows with missing values on 41099 rows
Removed 0 rows with missing values on 41099 rows
Removed 0 columns with missing values on 16 columns
New shape: (41099, 17)
Original task: classification for spotify
Classes (array([0, 1]), array([20551, 20548]))
X shape: (10000, 17), y shape: (10000,)


Num col names:  ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'chorus_hit']
Cat col names:  ['name', 'artist', 'key', 'mode', 'time_signature', 'sections']


/scratch/lgrinszt/micromamba/envs/lm_tab/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/scratch/lgrinszt/micromamba/envs/lm_tab/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Model No. xx: 100%|██████████| 2/2 [00:01<00:00,  1.78it/s]


out (500,)
out after softmax (500,)

The AUROC for CARTE: 0.7920

The accuracy for CARTE: 0.7080

The balanced accuracy for CARTE: 0.7050


Num col names:  ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'chorus_hit']
Cat col names:  ['name', 'artist', 'key', 'mode', 'time_signature', 'sections']


...

Model No. xx: 100%|██████████| 2/2 [00:01<00:00,  1.68it/s]


out (500,)
out after softmax (500,)

The AUROC for CARTE: 0.7687

The accuracy for CARTE: 0.6640

The balanced accuracy for CARTE: 0.6590


Num col names:  ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'chorus_hit']
Cat col names:  ['name', 'artist', 'key', 'mode', 'time_signature', 'sections']


Model No. xx: 100%|██████████| 2/2 [00:01<00:00,  1.72it/s]


out (500,)
out after softmax (500,)

The AUROC for CARTE: 0.7944

The accuracy for CARTE: 0.6900

The balanced accuracy for CARTE: 0.6897


In [5]:
y_pred_proba.shape

(500,)

In [15]:
y_pred_proba.shape

(10, 10)

In [14]:
y_pred.shape

(10, 10)

In [7]:
# Get the datasets where 'carte' encoding is present
carte_datasets = set(melted_results[melted_results['encoding'] == 'carte']['dataset'].unique())

# Get all unique datasets
all_datasets = set(melted_results['dataset'].unique())

# Find the datasets missing for 'carte'
missing_datasets_for_carte = all_datasets - carte_datasets

print("Datasets missing for 'carte' encoding:")
print(sorted(list(missing_datasets_for_carte)))


Datasets missing for 'carte' encoding:
['ae', 'airbnb', 'bikewale', 'channel', 'cloth', 'employee-remuneration-and-expenses-earning-over-75000', 'imdb', 'prod', 'qaa', 'qaq', 'salary', 'wine', 'wine_review']


In [79]:
valid_datasets

['clear_corpus',
 'company_employees',
 'employee_salary',
 'goodreads',
 'journal_jcr_cls',
 'ramen_ratings',
 'spotify',
 'us_accidents_counts',
 'us_accidents_severity',
 'us_presidential',
 'zomato',
 'jigsaw',
 'fake',
 'kick',
 'mercari',
 'jc',
 'pop',
 'book']

In [69]:
from auto_mm_bench.datasets import dataset_registry

print(dataset_registry.list_keys())  # list of all dataset names
dataset_name = 'product_sentiment_machine_hack'

train_dataset = dataset_registry.create(dataset_name, 'train')
test_dataset = dataset_registry.create(dataset_name, 'test')
print(train_dataset.data)
print(test_dataset.data)

['product_sentiment_machine_hack', 'jigsaw_unintended_bias', 'jigsaw_unintended_bias100K', 'google_qa_label', 'google_qa_answer_helpful', 'google_qa_answer_plausible', 'google_qa_answer_type_procedure', 'google_qa_answer_type_reason_explanation', 'google_qa_question_type_reason_explanation', 'google_qa_answer_satisfaction', 'women_clothing_review', 'melbourne_airbnb', 'mercari_price_suggestion', 'ae_price_prediction', 'mercari_price_suggestion100K', 'imdb_genre_prediction', 'fake_job_postings', 'kick_starter_funding', 'jc_penney_products', 'wine_reviews', 'news_popularity', 'news_channel', 'news_popularity2', 'fake_job_postings2', 'bookprice_prediction', 'data_scientist_salary', 'california_house_price']
      Unnamed: 0  Text_ID                                Product_Description  \
0           5743     2333  #techcrunch #google This Post Has Nothing to d...   
1           3042     3448  Data is the new oil. (Companies like Google an...   
2           4359      720  my sister is throwi

In [ ]:
train_dataset.data

,Unnamed: 0,Text_ID,Product_Description,Product_Type,Sentiment
0,5743,2333,#techcrunch #google This Post Has Nothing to d...,9,2
1,3042,3448,Data is the new oil. (Companies like Google an...,3,1
2,4359,720,my sister is throwing the Google sxsw party to...,3,3
3,5685,2328,Clear +succinct visions make for great UX (thi...,9,2
4,2078,4955,40% of Google Maps use is mobile marissamayer ...,9,3
...,...,...,...,...,...
5086,1096,3235,I'm eyeing the grilled cheese stand for after ...,9,2
5087,3519,2728,Let's make that a temp-to-hire position RT @me...,9,2
5088,212,1370,@mention -&gt; RT @mention New #UberSocial for...,9,3
5089,2786,2530,@mention Did you find out the hours of the app...,9,2


In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [ ]:
data_url = 'https://raw.githubusercontent.com/mli/ag-docs/main/knot_theory/'
train_data = TabularDataset(f'{data_url}train.csv')
train_data.head()

In [ ]:
label = 'signature'
train_data[label].describe()

In [ ]:
import os

cpu_count = os.cpu_count()
print(f"Number of CPUs: {cpu_count}")


In [ ]:
predictor = TabularPredictor(label=label).fit(train_data,
                                              num_gpus=1)

In [ ]:
test_data = TabularDataset(f'{data_url}test.csv')

y_pred = predictor.predict(test_data.drop(columns=[label]))
y_pred.head()

In [ ]:
predictor.evaluate(test_data, silent=True)

In [ ]:
predictor.leaderboard(test_data)

# Text

In [4]:
%matplotlib inline

import numpy as np
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(123)

In [5]:
from autogluon.core.utils.loaders import load_pd
train_data = load_pd.load('https://autogluon-text.s3-accelerate.amazonaws.com/glue/sst/train.parquet')
test_data = load_pd.load('https://autogluon-text.s3-accelerate.amazonaws.com/glue/sst/dev.parquet')
subsample_size = 1000  # subsample data for faster demo, try setting this to larger values
train_data = train_data.sample(n=subsample_size, random_state=0)
train_data.head(10)

,sentence,label
43787,very pleasing at its best moments,1
16159,", american chai is enough to make you put away...",0
59015,too much like an infomercial for ram dass 's l...,0
5108,a stirring visual sequence,1
67052,cool visual backmasking,1
35938,hard ground,0
49879,"the striking , quietly vulnerable personality ...",1
51591,pan nalin 's exposition is beautiful and myste...,1
56780,wonderfully loopy,1
28518,"most beautiful , evocative",1


In [8]:
data = load_pd.load("../data/spotify.parquet")
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

subsample_size = 5000  # subsample data for faster demo, try setting this to larger values
train_data = train_data.sample(n=subsample_size, random_state=0)



In [9]:
data

,name,artist,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,chorus_hit,sections,target
0,Jealous Kind Of Fella,Garland Green,0.417,0.620,3,-7.727,Major,0.0403,0.4900,0.000000,0.0779,0.8450,185.655,173533.0,3,32.94975,9,1
1,Initials B.B.,Serge Gainsbourg,0.498,0.505,3,-12.475,Major,0.0337,0.0180,0.107000,0.1760,0.7970,101.801,213613.0,4,48.82510,10,0
2,Melody Twist,Lord Melody,0.657,0.649,5,-13.392,Major,0.0380,0.8460,0.000004,0.1190,0.9080,115.940,223960.0,4,37.22663,12,0
3,Mi Bomba Sonó,Celia Cruz,0.590,0.545,7,-12.058,Minor,0.1040,0.7060,0.024600,0.0610,0.9670,105.592,157907.0,4,24.75484,8,0
4,Uravu Solla,P. Susheela,0.515,0.765,11,-3.515,Minor,0.1240,0.8570,0.000872,0.2130,0.9060,114.617,245600.0,4,21.79874,14,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41094,Lotus Flowers,Yolta,0.172,0.358,9,-14.430,Major,0.0342,0.8860,0.966000,0.3140,0.0361,72.272,150857.0,4,24.30824,7,0
41095,Calling My Spirit,Kodak Black,0.910,0.366,1,-9.954,Major,0.0941,0.0996,0.000000,0.2610,0.7400,119.985,152000.0,4,32.53856,8,1
41096,Teenage Dream,Katy Perry,0.719,0.804,10,-4.581,Major,0.0355,0.0132,0.000003,0.1390,0.6050,119.999,227760.0,4,20.73371,7,1
41097,Stormy Weather,Oscar Peterson,0.600,0.177,7,-16.070,Major,0.0561,0.9890,0.868000,0.1490,0.5600,120.030,213387.0,4,21.65301,14,0


In [10]:
train_data

,name,artist,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,chorus_hit,sections,target
35994,Come Back Song,Darius Rucker,0.516,0.745,9,-5.674,Major,0.0381,0.13800,0.000000,0.1140,0.770,177.781,235147.0,4,21.91768,10,1
9807,New Orleans,The Staple Singers,0.757,0.605,11,-11.869,Major,0.0427,0.10700,0.000476,0.0372,0.655,121.366,266307.0,4,32.71222,11,1
22905,Roadhouse Blues,Imperiet,0.554,0.911,9,-7.714,Major,0.0696,0.06190,0.000017,0.8720,0.607,127.148,248133.0,4,54.51084,11,0
11831,Fun House,The Stooges,0.521,0.745,9,-13.528,Major,0.0359,0.00364,0.000217,0.1240,0.914,131.694,465560.0,4,58.20880,19,0
30331,Gépinduló,Pokolgép,0.396,0.997,9,-5.180,Minor,0.2060,0.00287,0.883000,0.8030,0.123,162.022,287547.0,4,93.78882,8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,Brother Love's Travelling Salvation Show,Neil Diamond,0.428,0.432,8,-11.580,Major,0.0288,0.57500,0.009330,0.0888,0.808,135.142,211862.0,4,33.76803,11,1
6040,Reelin' And Rockin',The Dave Clark Five,0.538,0.794,7,-8.145,Major,0.0857,0.23600,0.000003,0.0850,0.950,91.110,167360.0,4,132.19797,4,1
27775,"Burn, Don't Freeze!",Sleater-Kinney,0.304,0.968,1,-4.450,Major,0.0976,0.24400,0.000003,0.2220,0.543,162.539,200720.0,4,18.08376,12,0
8263,Little Hands,Alexander 'Skip' Spence,0.330,0.485,8,-11.396,Minor,0.0367,0.19100,0.000000,0.0682,0.506,93.219,221627.0,4,40.41514,9,0


In [22]:
get_hyperparameter_config('multimodal')

{'NN_TORCH': {},
 'GBM': [{},
  {'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}},
  'GBMLarge'],
 'CAT': {},
 'XGB': {},
 'AG_AUTOMM': {},
 'VW': {}}

In [24]:
hyperparameters

{'NN_TORCH': {},
 'GBM': [{},
  {'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}},
  'GBMLarge'],
 'CAT': {},
 'XGB': {},
 'AG_AUTOMM': {},
 'VW': {},
 'model.hf_text.checkpoint_name': 'intfloat/e5-large-v2'}

In [11]:
from autogluon.multimodal import MultiModalPredictor
from autogluon.tabular.configs.hyperparameter_configs import get_hyperparameter_config
hyperparameters = get_hyperparameter_config('multimodal')
import uuid
from autogluon.tabular import TabularDataset, TabularPredictor
model_path = f"./tmp/{uuid.uuid4().hex}-automm_sst"
#predictor = MultiModalPredictor(label='target')
 #hyperparameters={"model.hf_text.checkpoint_name": "intfloat/e5-large-v2"})#, path=model_path)
predictor = TabularPredictor(label='target')
# hyperparameters={"model.hf_text.checkpoint_name": "intfloat/e5-large-v2"})
#predictor.fit(train_data, time_limit=180, num_gpus=1)#, hyperparameters=hyperparameters)
hyperparameters = get_hyperparameter_config('multimodal')
hyperparameters["AG_AUTOMM"]["model.hf_text.checkpoint_name"] = "intfloat/e5-large-v2"
print(hyperparameters)
predictor.fit(train_data, time_limit=60,
hyperparameters=hyperparameters)

No path specified. Models will be saved in: "AutogluonModels/ag-20240902_101310"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.10.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Wed Dec 22 13:25:12 UTC 2021
CPU Count:          48
Memory Avail:       174.26 GB / 187.29 GB (93.0%)
Disk Space Avail:   1119.08 GB / 49150.01 GB (2.3%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets.
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='best_quality'   : Maximize accuracy. Default time_limit=3600.
	presets='high_quality'   : Strong accuracy with fast inference speed. Default time_limit=3600.
	presets='good_quality'   : Good accuracy with very fast inference speed. Default time_limit=3600.
	presets='medium_quality' : Fast traini

{'NN_TORCH': {}, 'GBM': [{}, {'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}}, 'GBMLarge'], 'CAT': {}, 'XGB': {}, 'AG_AUTOMM': {'model.hf_text.checkpoint_name': 'intfloat/e5-large-v2'}, 'VW': {}}


			Fitting CategoryMemoryMinimizeFeatureGenerator...
		Fitting TextSpecialFeatureGenerator...
			Fitting BinnedFeatureGenerator...
			Fitting DropDuplicatesFeatureGenerator...
		Fitting TextNgramFeatureGenerator...
			Fitting CountVectorizer for text features: ['name']
			CountVectorizer fit with vocabulary size = 67
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        : 11 | ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', ...]
		('object', [])       :  5 | ['artist', 'key', 'mode', 'time_signature', 'sections']
		('object', ['text']) :  1 | ['name']
	Types of features in processed data (raw dtype, special dtypes):
		('category', [])                    :  4 | ['artist', 'key', 'time_signature', 'sections']
		('category', ['text_as_category'])  :  1 | ['name']
		('float', [])                       : 11 | ['

In [ ]:
{'accuracy': 0.8670316301703163, 'balanced_accuracy': 0.867213666381244, 'mcc': 0.7384568672961448, 'roc_auc': 0.9399838526195745, 'f1': 0.8734221192819919, 'precision': 0.8307997356245869, 'recall': 0.920654296875}


In [12]:
test_score = predictor.evaluate(test_data)
print(test_score)

{'accuracy': 0.8604622871046229, 'balanced_accuracy': 0.860630157500303, 'mcc': 0.7246103496448713, 'roc_auc': 0.9350217713763942, 'f1': 0.8666434135565632, 'precision': 0.8273029966703662, 'recall': 0.909912109375}


In [26]:
test_score = predictor.evaluate(test_data)
print(test_score)

Load pretrained checkpoint: /scratch/lgrinszt/lm_tab/AutogluonModels/ag-20240901_190357/models/MultiModalPredictor/automm_model/model.ckpt
SLURM auto-requeueing enabled. Setting signal handlers.


Predicting: |          | 0/? [00:00<?, ?it/s]

{'accuracy': 0.8653284671532847, 'balanced_accuracy': 0.8654847908205019, 'mcc': 0.733897057218192, 'roc_auc': 0.9375991599289223, 'f1': 0.8708736731599207, 'precision': 0.8338172883627429, 'recall': 0.911376953125}


In [30]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/soda/lgrinszt/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
predictor.column_types

In [ ]:
predictor.feature_metadata.type_group_map_special

In [13]:
test_score = predictor.evaluate(test_data, metrics=['acc', 'f1', 'roc_auc', "balanced_accuracy"])
print(test_score)

SLURM auto-requeueing enabled. Setting signal handlers.


Predicting: |          | 0/? [00:00<?, ?it/s]

{'acc': 0.8524330900243309, 'f1': 0.8626740631721952, 'roc_auc': 0.9144899236936227, 'balanced_accuracy': 0.8526970079867846}


In [ ]:
predictor.leaderboard(test_data)

In [ ]:
sentence1 = "it's a charming and often affecting journey."
sentence2 = "It's slow, very, very, very slow."
predictions = predictor.predict({'sentence': [sentence1, sentence2]})
print('"Sentence":', sentence1, '"Predicted Sentiment":', predictions[0])
print('"Sentence":', sentence2, '"Predicted Sentiment":', predictions[1])

In [ ]:
probs = predictor.predict_proba({'sentence': [sentence1, sentence2]})
print('"Sentence":', sentence1, '"Predicted Class-Probabilities":', probs[0])
print('"Sentence":', sentence2, '"Predicted Class-Probabilities":', probs[1])

In [ ]:
test_predictions = predictor.predict(test_data)
test_predictions.head()

In [ ]:
embeddings = predictor.extract_embedding(test_data)
print(embeddings.shape)

In [ ]:
from sklearn.manifold import TSNE
X_embedded = TSNE(n_components=2, random_state=123).fit_transform(embeddings)
for val, color in [(0, 'red'), (1, 'blue')]:
    idx = (test_data['label'].to_numpy() == val).nonzero()
    plt.scatter(X_embedded[idx, 0], X_embedded[idx, 1], c=color, label=f'label={val}')
plt.legend(loc='best')

# Test pipeline

In [ ]:
from src.data_loading import load_data
from skrub import MinHashEncoder
from sklearn.decomposition import PCA
from src.utils import FeaturesExtractor, FixedSizeSplit
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
#from tabpfn import TabPFNClassifier
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
import time
from sentence_transformers import SentenceTransformer
from src.encodings import encode_high_cardinality_features
from src.utils import run_on_encoded_data, FeaturesExtractor
from skrub import TableVectorizer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.random_projection import GaussianRandomProjection
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import submitit
from functools import partial
from itertools import product
import time
from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.multimodal import MultiModalPredictor
from autogluon.tabular.configs.hyperparameter_configs import get_hyperparameter_config


/scratch/lgrinszt/micromamba/envs/lm_tab/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning:

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)

/scratch/lgrinszt/micromamba/envs/lm_tab/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning:

Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.



In [ ]:
%cd scripts

[Errno 2] No such file or directory: 'scripts'
/scratch/lgrinszt/lm_tab/scripts


In [ ]:
len(pd.read_parquet("../data/wine.parquet")["target"].unique())

30

In [ ]:
datasets = ['bikewale', 'clear_corpus', 'company_employees',
       'employee-remuneration-and-expenses-earning-over-75000',
       'employee_salary', 'goodreads', 'journal_jcr_cls', 'ramen_ratings',
       'spotify', 'us_accidents_counts', 'us_accidents_severity',
       'us_presidential', 'wine_review', 'zomato']
datasets.extend(['prod',
 'airbnb',
 'channel',
 'wine',
 'imdb',
 'jigsaw',
 'fake',
 'kick',
 'ae',
 'qaa',
 'qaq',
 'cloth',
 'mercari',
 'jc',
 'pop',
 'book',
 'salary',
 'house']
)
datasets_not_binary = []

for dataset in datasets:
    X, y = load_data(dataset, max_rows=10000)
    if len(np.unique(y)) > 2:
        print(dataset, np.unique(y, return_counts=True))
        datasets_not_binary.append(dataset)

Removed 0 columns with missing values on 8 columns
Removed 2 rows with missing values on 9003 rows
Removed 2 rows with missing values on 9001 rows
Removed 0 columns with missing values on 7 columns
New shape: (9001, 8)
Original task: regression for bikewale
Converting to binary classification
Classes (array([0, 1]), array([4645, 4356]))
X shape: (9001, 8), y shape: (9001,)
Removed 0 columns with missing values on 18 columns
Removed 278 rows with missing values on 4724 rows
Removed 278 rows with missing values on 4446 rows
Removed 0 columns with missing values on 17 columns
New shape: (4446, 18)
Original task: regression for clear_corpus
Converting to binary classification
Classes (array([0, 1]), array([2223, 2223]))
X shape: (4446, 18), y shape: (4446,)
Removed 0 columns with missing values on 6 columns
Removed 1378 rows with missing values on 4769 rows
Removed 1378 rows with missing values on 3391 rows
Removed 0 columns with missing values on 5 columns
New shape: (3391, 6)
Original ta

Removed 0 columns with missing values on 5 columns
Removed 2410 rows with missing values on 35396 rows
Removed 2410 rows with missing values on 32986 rows
Removed 0 columns with missing values on 4 columns
New shape: (32986, 5)
Original task: regression for employee-remuneration-and-expenses-earning-over-75000
Converting to binary classification
Classes (array([0, 1]), array([16493, 16493]))
X shape: (10000, 5), y shape: (10000,)
Removed 0 columns with missing values on 8 columns
Removed 17 rows with missing values on 9228 rows
Removed 17 rows with missing values on 9211 rows
Removed 0 columns with missing values on 7 columns
New shape: (9211, 8)
Original task: regression for employee_salary
Converting to binary classification
Classes (array([0, 1]), array([4606, 4605]))
X shape: (9211, 8), y shape: (9211,)
Removed 2 columns with missing values on 12 columns
Removed 1652 rows with missing values on 3967 rows
Removed 1652 rows with missing values on 2315 rows
Removed 2 columns with miss

In [ ]:
datasets_not_binary

['wine_review',
 'prod',
 'airbnb',
 'channel',
 'wine',
 'ae',
 'qaa',
 'qaq',
 'cloth',
 'salary']

In [ ]:
dataset = "cloth"
n_test = 500
n_train = 500
features = "all"

X, y = load_data(dataset, max_rows=10000)
if len(X) < n_train + n_test:
    print("Not enough data")

Removed 0 columns with missing values on 6 columns
Removed 3074 rows with missing values on 18788 rows
Removed 3074 rows with missing values on 15714 rows
Removed 0 columns with missing values on 5 columns
New shape: (15714, 6)
Original task: classification for cloth
More than 2 classes, converting to binary classification
Classes (array([0, 1, 2, 3, 4]), array([ 564, 1074, 2002, 3467, 8607]))
X shape: (10000, 6), y shape: (10000,)


In [ ]:
y

array([4, 4, 2, ..., 3, 4, 2])

In [ ]:
encoding = "skrub__minhash_30"
X_enc, X_rest = encode_high_cardinality_features(X, encoding, dataset_name=dataset, override_cache=False, cardinality_threshold=30, fail_if_not_cached=True)
cv = FixedSizeSplit(n_splits=1, n_train=n_train, n_test=n_test, random_state=42)

numeric ['Age']
low_cardinality ['Division Name', 'Department Name', 'Class Name']
high_cardinality ['Title', 'Review Text']
High cardinality columns ['Title', 'Review Text']
working dir /scratch/lgrinszt/lm_tab/scripts
Loaded from cache
working dir /scratch/lgrinszt/lm_tab/scripts
Loaded from cache


In [ ]:
encoding

'skrub__minhash_30'

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
res = run_on_encoded_data(X_enc, None, y, "passthrough", "passthrough", "GradientBoostingClassifier", GradientBoostingClassifier(), encoding, cv, dataset=dataset, features="text_only")

Running GradientBoostingClassifier with passthrough and skrub__minhash_30
[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89], [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]]
3
[('dim_reduction_0', 'passthrough', [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]), ('dim_reduction_1', 'passthrough', [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]), ('dim_reduction_2', 'passthrough', [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59])]
(4446, 90)


ValueError: Found input variables with inconsistent numbers of samples: [4446, 1000]

In [ ]:
np.mean(res["roc_auc"])

0.7082036572757847

In [ ]:
from skrub import TableVectorizer
enc = TableVectorizer()
X_enc = enc.fit_transform(X_enc)

In [ ]:
import os
del os.environ["SLURM_NTASKS"]
del os.environ["SLURM_JOB_NAME"]

In [ ]:
# Prepare the data for AutoGluon
data = pd.DataFrame(X)
data['target'] = y

all_scores = []

# Use cv to split the data and fit the model
for train_idx, test_idx in cv.split(X_enc):
    predictor = TabularPredictor(label='target')
    train_data = data.iloc[train_idx]
    test_data = data.iloc[test_idx]
    
    # Fit the model using the training data
    predictor.fit(train_data=train_data, time_limit=180, num_gpus=1,
        hyperparameters = get_hyperparameter_config('multimodal'))
    
    # Evaluate the model using the test data
    performance = predictor.evaluate(test_data)
    print("Model performance:", performance)
    all_scores.append(performance)



No path specified. Models will be saved in: "AutogluonModels/ag-20240830_122402"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.10.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Wed Dec 22 13:25:12 UTC 2021
CPU Count:          48
Memory Avail:       168.33 GB / 187.29 GB (89.9%)
Disk Space Avail:   2385.44 GB / 49150.01 GB (4.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets.
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='best_quality'   : Maximize accuracy. Default time_limit=3600.
	presets='high_quality'   : Strong accuracy with fast inference speed. Default time_limit=3600.
	presets='good_quality'   : Good accuracy with very fast inference speed. Default time_limit=3600.
	presets='medium_quality' : Fast traini

Model performance: {'accuracy': 0.576, 'balanced_accuracy': 0.25840336134453784, 'mcc': 0.21387916452545758}


In [ ]:
np.mean([score['roc_auc'] for score in all_scores])

0.704097553052583

In [ ]:
[score['roc_auc'] for score in all_scores]

[0.7013759702354224,
 0.720768,
 0.6885599009821414,
 0.7137916126242975,
 0.71770993343574,
 0.7079197617827869,
 0.6785576923076924]